# Chapter 15 &mdash; PCP is Undecidable, via Computational Histories

**Concept 2 of the Chapter 15 decomposition:** *PCP is Undecidable, via the Computational History Method*

Manufacture an instance whose only solution <i>is</i> an accepting computation history of $M$ on $w$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter15/Concept-PCP-Undecidable/Concept-PCP-Undecidable.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
from jove.Def_TM         import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The proof strategy is the **computational history method**, and it is worth learning as
a technique rather than a one-off trick.

Given $\langle M,w\rangle$, build a PCP instance whose dominoes can only be assembled
into a matching pair if the string they spell out is the **accepting computation
history** of $M$ on $w$ &mdash; the sequence of IDs
$$\#\,C_0\,\#\,C_1\,\#\,\cdots\,\#\,C_{accept}\,\#$$

The **bottom** row always runs one configuration ahead of the **top**. The only way to
keep them matching is to extend the bottom by a *legal successor* configuration &mdash; so
the dominoes enforce the transition relation, one cell at a time.

Then: a solution exists $\iff$ $M$ accepts $w$. A PCP decider would decide $A_{TM}$,
which is impossible.

## 2. Definitions

### Computation histories, concretely

In [ ]:
def history(T, tape, fuel=60):
    trunc, halts = run_tm(T, tape if tape else '.', fuel, chatty=False)
    if not halts: return None
    cfg, path = halts[0]
    def compact(c):
        q, head, tp, f = c
        tp = tp.rstrip('.') or '.'
        if head >= len(tp): tp = tp + '.' * (head - len(tp) + 1)
        return tp[:head] + q + tp[head:]
    return [compact(c) for c in path] + [compact(cfg)]

### The 'bottom runs one ahead' invariant

In [ ]:
# --- a Post Correspondence solver, bounded by tile count ----------------
# An instance is a list of (top, bottom) dominoes.  A solution is a
# non-empty sequence of indices whose concatenated tops equal its bottoms.
def pcp_search(tiles, maxlen=8):
    from collections import deque
    # a partial solution is (indices, top, bottom); one side is a prefix
    # of the other, or the partial is dead
    dq = deque([((i,), t, b) for i, (t, b) in enumerate(tiles)])
    while dq:
        idx, top, bot = dq.popleft()
        if top == bot:
            return list(idx)
        if len(idx) >= maxlen:
            continue
        if not (top.startswith(bot) or bot.startswith(top)):
            continue                                  # dead: they diverge
        for j, (t, b) in enumerate(tiles):
            dq.append((idx + (j,), top + t, bot + b))
    return None

def pcp_check(tiles, sol):
    top = ''.join(tiles[i][0] for i in sol)
    bot = ''.join(tiles[i][1] for i in sol)
    return top == bot, top, bot

def show_tiles(tiles):
    print("   " + "  ".join("[%s/%s]" % t for t in tiles))


def bottom_ahead(tiles, sol):
    # show the partial tops and bottoms as the solution is assembled
    top = bot = ''
    rows = []
    for i in sol:
        top += tiles[i][0]; bot += tiles[i][1]
        rows.append((top, bot, len(bot) - len(top)))
    return rows

## 3. Tests

A computation history is just the list of IDs.

In [ ]:
Flip = md2mc('''TM
I : 0 ; 1 , R -> I
I : 1 ; 0 , R -> I
I : . ; . , S -> F
''')
h = history(Flip, '01')
print("accepting computation history of Flip on '01' :")
print("   # " + " # ".join(h) + " #")
assert h and h[-1][0] == 'F' or 'F' in h[-1]

The encoded string the PCP instance must spell out.

In [ ]:
enc = '#' + '#'.join(h) + '#'
print("encoded :", enc)
print("length  :", len(enc))
print("\nThe PCP instance is built so that THIS string, and only strings")
print("of this shape, can appear as the matched top and bottom.")

**The bottom runs one configuration ahead**, which forces legality.

In [ ]:
CLASSIC = [('b', 'ca'), ('a', 'ab'), ('ca', 'a'), ('abc', 'c')]
sol = pcp_search(CLASSIC, maxlen=6)
for top, bot, lead in bottom_ahead(CLASSIC, sol):
    print("  top %-10r bottom %-10r bottom leads by %d" % (top, bot, lead))
print("\nIn the TM reduction that lead is exactly one configuration, and the")
print("only dominoes that can close the gap are the ones encoding a legal move.")

So a solution exists **iff** $M$ accepts $w$.

In [ ]:
print("M accepts w   =>  the history exists  =>  the dominoes can be laid out")
print("dominoes laid =>  the matched string IS a valid accepting history")
print("              =>  M accepts w")
print()
print("A PCP decider would therefore decide A_TM.  A_TM is undecidable.")
print("Therefore PCP is undecidable.")

The technique generalises; that is the real lesson.

In [ ]:
USES = ["PCP undecidability (this proof)",
        "CFG ambiguity undecidability (Concept 4)",
        "LBA emptiness undecidability",
        "tiling problems, and the domino problem of the plane"]
for u in USES: print("  *", u)
print("\nWhenever you must force a structure to encode a COMPUTATION, reach")
print("for computation histories.")

## 4. Exercises


1. Write out the history of `Flip` on `'10'`. How many configurations?
2. Why must the bottom lead rather than the top?
3. Which part of the construction handles the *first* configuration $C_0$?

In [ ]:
# Your work for the exercises above.